# Required Setup

### Estimated setup time: 5 minutes

This notebook prepares your Databricks environment for the Finance Genie demo.
It creates the required catalog, schema, and volume; generates the synthetic
financial fraud dataset in Delta Lake; and stores Neo4j credentials as Databricks secrets.

**Run this notebook once before starting any other notebooks.**

### Prerequisites

- A **Dedicated** compute cluster with the Neo4j Spark Connector installed:
  - Access mode: **Dedicated** (required for the Spark Connector)
  - Runtime: 13.3 LTS or higher
  - Maven library: `org.neo4j:neo4j-connector-apache-spark_2.12:5.3.1_for_spark_3`
- A running **Neo4j Aura** instance with connection details ready
- Permission to **create a catalog** in your workspace (or an existing catalog you can use)

## Step 1: Enter Your Neo4j Connection Details

Fill in the widgets at the top of this notebook with your Neo4j connection information.
These values will be stored as Databricks secrets in scope `neo4j-graph-engineering`
so all subsequent notebooks can read them automatically.

### Where Are My Neo4j Credentials Stored?

Your Neo4j connection details are saved as Databricks secrets in the scope
**`neo4j-graph-engineering`** with keys `uri`, `username`, and `password`.
All subsequent notebooks read from this scope automatically via `dbutils.secrets.get`.

In [ ]:
dbutils.widgets.text('neo4j_url', '', 'Neo4j URI (e.g. neo4j+s://xxx.databases.neo4j.io)')
dbutils.widgets.text('neo4j_username', 'neo4j', 'Neo4j Username')
dbutils.widgets.text('neo4j_password', '', 'Neo4j Password')

## Step 2: Run the Setup

The cells below will:
1. Create a Unity Catalog catalog, schema, and volume based on your username
2. Generate the synthetic financial fraud dataset and write it to Delta tables
3. Store your Neo4j credentials as Databricks secrets
4. Verify the Neo4j connection

Review the output to confirm everything succeeded.

In [ ]:
neo4j_url      = dbutils.widgets.get('neo4j_url')
neo4j_username = dbutils.widgets.get('neo4j_username')
neo4j_password = dbutils.widgets.get('neo4j_password')

if not neo4j_url or not neo4j_password:
    raise ValueError(
        'Please fill in the Neo4j URI and Password widgets at the top of this notebook before running.'
    )

In [ ]:
%run ./Includes/config

In [ ]:
%run ./Includes/_lib/setup_orchestrator.py

In [ ]:
catalog_config = CONFIG['catalog']
secrets_config = CONFIG['secrets']

print('Configuration loaded:')
print(f"  Catalog prefix: {catalog_config['prefix']}")
print(f"  Schema:         {catalog_config['schema_name']}")
print(f"  Volume:         {catalog_config['volume_name']}")
print(f"  Secret scope:   {secrets_config['scope_name']}")

In [ ]:
# Step 1: Create catalog, schema, and volume
username     = get_username()
catalog_name = derive_catalog_name(catalog_config['prefix'], username)

catalog_info = setup_catalog_and_schema(
    catalog_name=catalog_name,
    schema_name=catalog_config['schema_name'],
    volume_name=catalog_config['volume_name'],
)

In [ ]:
# Step 2: Generate synthetic fraud dataset and write to Delta tables
table_counts = generate_and_load_data(
    catalog_name=catalog_info['catalog'],
    schema_name=catalog_info['schema'],
    data_config=CONFIG['data'],
)

In [ ]:
# Step 3: Store Neo4j credentials as Databricks secrets
setup_neo4j_secrets(
    scope_name=secrets_config['scope_name'],
    neo4j_url=neo4j_url,
    neo4j_username=neo4j_username,
    neo4j_password=neo4j_password,
)

In [ ]:
# Step 4: Verify Neo4j connection
neo4j_connected = verify_neo4j_connection(neo4j_url, neo4j_username, neo4j_password)

In [ ]:
print_summary({
    'catalog':         catalog_info['catalog'],
    'schema':          catalog_info['schema'],
    'volume':          catalog_info['volume'],
    'table_counts':    table_counts,
    'neo4j_url':       neo4j_url,
    'scope_name':      secrets_config['scope_name'],
    'neo4j_connected': neo4j_connected,
})

## Next Steps

If the setup completed successfully, proceed to **01_neo4j_ingest** to push the
synthetic fraud dataset into Neo4j via the Spark Connector.